In [1]:
# Imports.
import numpy as np;
import xarray as xr;
import matplotlib.pyplot as plt;
import h5_reader_xr as reader;
import gysela_utilities as gys_utils;
import phi2D_utilities as utils;
import phi2D_analytical as analytical;

# Styling.
plt.style.use("ggplot");

In [ ]:
# Test inputs - delete when done!
perturbed_case_directory_path = "/zhisongqu_data/seth/GYSELA/raw/batch_11/DN_DELTASCAN_A3_PERTURB_-0.3";
quiescent_case_directory_path = "/zhisongqu_data/seth/GYSELA/raw/batch_11/DN_DELTASCAN_A3_QUIESCENT_-0.3";
perturbed_case_data_arrays = reader.fetch_phi2D_data(perturbed_case_directory_path, parallelise = True);
quiescent_case_data_arrays = reader.fetch_phi2D_data(quiescent_case_directory_path, parallelise = True);
dt_diag = reader.fetch_dt_diag(perturbed_case_directory_path);
jacobian_dictionary = reader.fetch_jacobian(perturbed_case_directory_path);

Found 1201 files in the directory. Loading with 16 workers...


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.1s
[Parallel(n_jobs=16)]: Done 512 tasks      | elapsed:    1.5s
[Parallel(n_jobs=16)]: Done 1170 out of 1201 | elapsed:    3.6s remaining:    0.1s
[Parallel(n_jobs=16)]: Done 1201 out of 1201 | elapsed:    3.8s finished


Finished compiling data from all files in the directory.


In [ ]:
def plot_rosenbluth_hinton(perturbed_phi2D_list, quiescent_phi2D_list, dt_diag, jacobian_dictionary, aspect_ratio, safety_factor, title_suffix = "", effective_radius = 0.7, residual_window = 100):
	
	# Signal isolation and data processing.
	perturbed_radial_time_series = utils.generate_poloidally_averaged_time_series(perturbed_phi2D_list, jacobian_dictionary, effective_radius);
	quiescent_radial_time_series = utils.generate_poloidally_averaged_time_series(quiescent_phi2D_list, jacobian_dictionary, effective_radius);
	radial_time_series = perturbed_radial_time_series - quiescent_radial_time_series;
	# We recover the actual time in code units here (normalised to the ion cyclotron frequency).
	# This does not actually change the size of the array, only its spacing.
	time_range = gys_utils.generate_time_range_by_series(radial_time_series, dt_diag);
	GAM_frequency = utils.extract_gam_frequency(perturbed_phi2D_list, dt_diag, jacobian_dictionary, effective_radius);
	amplitude_envelope = utils.generate_damping_envelope(radial_time_series.values, GAM_frequency, dt_diag);
	# The filtered signal can sometimes have an initial 'transient' period.
	# As such, phi(t = 0) is taken to be the maximum amplitude of the signal.
	initial_amplitude = max(amplitude_envelope);
	residual_level = utils.calculate_residual_level(amplitude_envelope, residual_window, use_heuristic_approach = True);
	residual_level_xc2006 = analytical.residual_XiaoCatto2006(1. / aspect_ratio, safety_factor, k_rho = 0.04) * initial_amplitude;
	residual_level_rh1998 = analytical.residual_RosenbluthHinton1998(1. / aspect_ratio, safety_factor) * initial_amplitude;

	# Figure plotting logic.
	plt.figure(figsize = (10, 5));
	plt.plot(time_range, radial_time_series, color = "red", lw = 2.5, label = "Signal");
	plt.plot(time_range, amplitude_envelope, color = "black", linestyle = "--", label= "Damping envelope", lw = 2.0);
	plt.axhline(y = residual_level_xc2006, color = "blue", linestyle = ":", label = "Xiao-Catto 2006", lw = 2.0);
	plt.axhline(y = residual_level_rh1998, color = "firebrick", linestyle = ":", label = "Rosenbluth-Hinton 1998", lw = 2.0);
	plt.axhline(y = residual_level, color = "gray", linestyle = ":", label = "Actual residual", lw = 2.0);
	plt.xlabel("t [GYSELA code units]");
	plt.ylabel(r"$\langle \Phi \rangle_\theta$");
	plt.xlim(left = 50, right = 40000);
	plt.title(f"GAM damping & residual relaxation, q = 3.5, $\\rho$ = {effective_radius}, {title_suffix}");
	plt.legend(frameon = True, loc = "lower right");
	plt.show();

plot_rosenbluth_hinton(
	perturbed_case_data_arrays, 
	quiescent_case_data_arrays, 
	dt_diag = 25, 
	jacobian_dictionary = jacobian_dictionary, 
	aspect_ratio = 3.0, 
	safety_factor = 3.5, 
	effective_radius = 0.42, 
	residual_window = 100, 
	title_suffix = r"A = 3, Culham, $\delta$ = -0.2, trend-subtracted");

TypeError: 'NoneType' object is not iterable